In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from biomae.paths import get_project_root

import os
import copy
import time
import json
import random
import shutil

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    balanced_accuracy_score,
)

# ================= CONFIGURATION & HYPERPARAMÈTRES =================

# Mise à jour du chemin racine vers votre nouveau projet
BASE_PATH = get_project_root()

# Chemin direct vers le dataset traité (contient train/ et val/)
DATA_DIR = os.path.join(BASE_PATH, "data/02_processed/n_mobilenet_mfic_dataset")

# Nom de l'expérience pour les logs (output)
RUN_NAME = "mobilenet_v3_mfic_4classes_v4"
OUTPUT_DIR = os.path.join(BASE_PATH, "outputs", RUN_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Paramètres d'entraînement
BATCH_SIZE = 4
NUM_EPOCHS = 150
INPUT_SIZE = 512
NUM_WORKERS = 4
SEED = 42

# Fine-tuning
UNFREEZE_LAST_N_FEATURE_BLOCKS = 3
LR_HEAD = 5e-4
LR_BACKBONE = 1e-4
WEIGHT_DECAY = 1e-4

# Gestion du déséquilibre
USE_WEIGHTED_SAMPLER = True
USE_WEIGHTED_CE = False

# Règle de décision à seuils (Logique métier)
THRESHOLD_FEMALE = 0.40
THRESHOLD_INDET = 0.50
THRESHOLD_COUPLE = 0.50
USE_ARGMAX_DECISION = False

# Régularisation
USE_LABEL_SMOOTHING = True
LABEL_SMOOTHING = 0.05

# Normalisation standard ImageNet (DOIT être identique en inférence)
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

# ================= OUTILS & UTILITAIRES =================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def plot_history(history, save_path):
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    ax[0].plot(history["train_loss"], label="Train Loss")
    ax[0].plot(history["val_loss"], label="Val Loss")
    ax[0].set_title("Evolution de la Loss")
    ax[0].set_xlabel("Epochs")
    ax[0].legend(); ax[0].grid(True)

    ax[1].plot(history["train_acc"], label="Train Acc")
    ax[1].plot(history["val_acc"], label="Val Acc")
    ax[1].plot(history["val_bal_acc"], label="Val Balanced Acc")
    ax[1].plot(history["val_f1_female"], label="Val F1 (femelle)")
    ax[1].set_title("Evolution des Métriques")
    ax[1].set_xlabel("Epochs")
    ax[1].legend(); ax[1].grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "results.png"))
    plt.close()

def plot_confusion_matrix(y_true, y_pred, classes, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    cm_norm = cm.astype(np.float64) / np.maximum(cm.sum(axis=1, keepdims=True), 1.0)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.title("Matrice de Confusion (Quantités)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "confusion_matrix.png"))
    plt.close()

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0.0, vmax=1.0, xticklabels=classes, yticklabels=classes)
    plt.title("Matrice de Confusion (Normalisée)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "confusion_matrix_normalized.png"))
    plt.close()

def build_weighted_sampler(image_folder_dataset):
    targets = [y for _, y in image_folder_dataset.samples]
    class_count = np.bincount(targets)
    class_weight = 1.0 / np.maximum(class_count, 1)
    sample_weights = np.array([class_weight[t] for t in targets], dtype=np.float64)
    sample_weights = torch.from_numpy(sample_weights)
    sampler = torch.utils.data.WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    return sampler, class_count

class ImageFolderWithPaths(datasets.ImageFolder):
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        path, _ = self.samples[index]
        return img, label, path

def predict_with_thresholds_4(probs, idx_f, idx_m, idx_i, idx_c, thr_f, thr_i, thr_c, labels_like):
    p_f, p_i, p_c = probs[:, idx_f], probs[:, idx_i], probs[:, idx_c]
    pred = torch.full_like(labels_like, fill_value=idx_m)
    pred = torch.where(p_f >= thr_f, torch.full_like(pred, idx_f), pred)
    pred = torch.where(p_c >= thr_c, torch.full_like(pred, idx_c), pred)
    pred = torch.where(p_i >= thr_i, torch.full_like(pred, idx_i), pred)
    return pred

def save_metadata_json(output_dir, class_names, input_size, norm_mean, norm_std):
    """Génère le fichier JSON critique pour l'inférence future."""
    metadata = {
        "model_architecture": "mobilenet_v3_large",
        "description": "Classification Gammare (Male, Femelle, Couple, Indeterminee)",
        "input_config": {
            "input_shape": [1, input_size, input_size, 3],
            "data_format": "channels_last",
            "preprocessing": {
                "mean": norm_mean,
                "std": norm_std,
                "note": "Input should be normalized: (pixel/255.0 - mean) / std"
            }
        },
        "labels_map": {
            "indices": {i: name for i, name in enumerate(class_names)},
            "names": class_names
        },
        "thresholds_used_in_training_val": {
            "female": THRESHOLD_FEMALE,
            "indet": THRESHOLD_INDET,
            "couple": THRESHOLD_COUPLE
        },
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    
    json_path = os.path.join(output_dir, "model_meta.json")
    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=4)
    print(f"[INFO] JSON de métadonnées généré : {json_path}")

# ================= FONCTION PRINCIPALE (MAIN) =================

def main():
    set_seed(SEED)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Utilisation du device : {device}")

    # 1) Augmentation de données
    data_transforms = {
        "train": transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(NORM_MEAN, NORM_STD),
        ]),
        "val": transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(NORM_MEAN, NORM_STD),
        ]),
    }

    # 2) Chargement des Datasets
    image_datasets = {x: ImageFolderWithPaths(os.path.join(DATA_DIR, x), data_transforms[x]) for x in ["train", "val"]}
    class_names = image_datasets["train"].classes
    dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "val"]}

    print(f"[INFO] Classes détectées (ORDRE DU JSON) : {class_names}")
    
    # Indices Mapping
    class_to_idx = image_datasets["train"].class_to_idx
    IDX_F, IDX_M = class_to_idx.get("femelle"), class_to_idx.get("male")
    IDX_I, IDX_C = class_to_idx.get("indeterminee"), class_to_idx.get("couple")

    if any(v is None for v in [IDX_F, IDX_M, IDX_I, IDX_C]):
        raise ValueError(f"ERREUR CRITIQUE : Classes manquantes. Trouvé : {list(class_to_idx.keys())}")

    # 3) Sampler
    train_sampler = None
    if USE_WEIGHTED_SAMPLER:
        train_sampler, class_count = build_weighted_sampler(image_datasets["train"])

    # 4) DataLoaders
    dataloaders = {
        "train": torch.utils.data.DataLoader(image_datasets["train"], batch_size=BATCH_SIZE, shuffle=(train_sampler is None), sampler=train_sampler, num_workers=NUM_WORKERS, pin_memory=True),
        "val": torch.utils.data.DataLoader(image_datasets["val"], batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
    }

    # 5) Modèle
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
    for p in model.parameters(): p.requires_grad = False
    if UNFREEZE_LAST_N_FEATURE_BLOCKS > 0:
        for p in model.features[-UNFREEZE_LAST_N_FEATURE_BLOCKS:].parameters(): p.requires_grad = True

    with torch.no_grad():
        dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
        backbone_out = torch.flatten(model.avgpool(model.features(dummy)), 1).shape[1]

    model.classifier = nn.Sequential(
        nn.Linear(backbone_out, 512),
        nn.LayerNorm(512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(512, len(class_names)),
    )
    model = model.to(device)

    # 6 & 7) Loss & Optimizer
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    head_params = list(model.classifier.parameters())
    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and "classifier" not in n]
    optimizer = optim.AdamW([{"params": backbone_params, "lr": LR_BACKBONE}, {"params": head_params, "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=8)

    # 8) Boucle d'entraînement
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_score = -1.0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_bal_acc": [], "val_f1_female": []}

    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        print("-" * 40)

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()
            running_loss, running_corrects, seen = 0.0, 0, 0
            y_true_epoch, y_pred_epoch = [], []

            for inputs, labels, _ in dataloaders[phase]:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                bs = inputs.size(0)
                seen += bs
                running_loss += loss.item() * bs

                if phase == "train":
                    _, preds = torch.max(outputs, 1)
                else:
                    probs = torch.softmax(outputs, dim=1)
                    if USE_ARGMAX_DECISION:
                        _, preds = torch.max(outputs, 1)
                    else:
                        preds = predict_with_thresholds_4(probs, IDX_F, IDX_M, IDX_I, IDX_C, THRESHOLD_FEMALE, THRESHOLD_INDET, THRESHOLD_COUPLE, labels)

                running_corrects += torch.sum(preds == labels.data).item()
                y_true_epoch.extend(labels.cpu().tolist())
                y_pred_epoch.extend(preds.cpu().tolist())

            epoch_loss = running_loss / max(seen, 1)
            epoch_acc = running_corrects / max(seen, 1)

            if phase == "train":
                history["train_loss"].append(epoch_loss)
                history["train_acc"].append(epoch_acc)
                print(f"Train | Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
            else:
                val_bal_acc = balanced_accuracy_score(y_true_epoch, y_pred_epoch)
                val_f1_female = f1_score(np.array(y_true_epoch) == IDX_F, np.array(y_pred_epoch) == IDX_F, zero_division=0)
                
                history["val_loss"].append(epoch_loss)
                history["val_acc"].append(epoch_acc)
                history["val_bal_acc"].append(val_bal_acc)
                history["val_f1_female"].append(val_f1_female)

                print(f"Val   | Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Bal_Acc: {val_bal_acc:.4f} F1_Fem: {val_f1_female:.4f}")
                scheduler.step(val_f1_female)

                if val_f1_female > best_score:
                    best_score = val_f1_female
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(best_model_wts, os.path.join(OUTPUT_DIR, "best_model.pth"))
                    print(">>> Meilleur modèle sauvegardé")

    time_elapsed = time.time() - since
    print(f"\nEntraînement terminé en {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")

    # 9) Évaluation Finale
    print("\n--- ANALYSE DES ERREURS ---")
    model.load_state_dict(best_model_wts)
    model.eval()
    
    y_true, y_pred, y_probs, val_paths = [], [], [], []
    with torch.no_grad():
        for inputs, labels, paths in dataloaders["val"]:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            if USE_ARGMAX_DECISION:
                _, preds = torch.max(outputs, 1)
            else:
                preds = predict_with_thresholds_4(probs, IDX_F, IDX_M, IDX_I, IDX_C, THRESHOLD_FEMALE, THRESHOLD_INDET, THRESHOLD_COUPLE, labels.to(device))
            
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())
            y_probs.extend(probs.cpu().tolist())
            val_paths.extend(paths)

    plot_confusion_matrix(y_true, y_pred, class_names, OUTPUT_DIR)
    plot_history(history, OUTPUT_DIR)
    with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w") as f:
        f.write(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    # Export des erreurs
    errors_dir = os.path.join(OUTPUT_DIR, "errors_val_visualisation")
    if os.path.exists(errors_dir): shutil.rmtree(errors_dir)
    os.makedirs(errors_dir)
    
    errors_indices = np.where(np.array(y_true) != np.array(y_pred))[0]
    y_probs_np = np.array(y_probs)
    for i in errors_indices:
        src, true_cls, pred_cls = val_paths[i], class_names[y_true[i]], class_names[y_pred[i]]
        conf = y_probs_np[i][y_pred[i]] * 100
        shutil.copy2(src, os.path.join(errors_dir, f"VRAI-{true_cls}_PRED-{pred_cls}_conf{conf:.1f}.jpg"))

    # 10) GÉNÉRATION DU JSON MANQUANT (NOUVEAU BLOC)
    save_metadata_json(OUTPUT_DIR, class_names, INPUT_SIZE, NORM_MEAN, NORM_STD)

    print(f"Terminé. Résultats : {OUTPUT_DIR}")

if __name__ == "__main__":
    main()
